In [0]:
%sql
create volume if not exists workspace.gtfs_bogota.staging_gtfs;

In [0]:
%sql
show volumes in workspace.gtfs_bogota;

In [0]:
# BLOQUE 1 — Parámetros y referencia al snapshot Raw

import re

catalog = "workspace"
schema = "gtfs_bogota"
raw_volume_path = f"/Volumes/{catalog}/{schema}/raw_gtfs"
staging_volume_path = f"/Volumes/{catalog}/{schema}/staging_gtfs"

# El Job recibe estos valores desde la tarea Raw. Los defaults permiten
# reproducir interactivamente el snapshot ya validado de octubre de 2025.
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()

if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", feed_version):
    raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
if not re.fullmatch(r"[0-9a-f]{64}", snapshot_sha256):
    raise ValueError("snapshot_sha256 debe ser un hash SHA-256 de 64 caracteres.")

raw_zip_path = (
    f"{raw_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}/gtfs.zip"
)

print(f"ZIP Raw: {raw_zip_path}")
print(f"Staging: {staging_volume_path}")

display(dbutils.fs.ls(raw_zip_path.rsplit('/', 1)[0]))

In [0]:
# BLOQUE 2 — Extracción idempotente de archivos GTFS hacia Staging

from datetime import datetime, timezone
import json
import shutil
from pathlib import PurePosixPath
import uuid
import zipfile

from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType,
)

# Staging conserva una copia de trabajo derivada del ZIP Raw.
# La versión y el hash mantienen el vínculo con la evidencia original.
staging_snapshot_path = (
    f"{staging_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}"
)

# Cada extracción tiene su propio identificador para auditoría.
extraction_run_id = str(uuid.uuid4())
extracted_at_utc = datetime.now(timezone.utc)

dbutils.fs.mkdirs(staging_snapshot_path)

# Evitamos reescribir archivos que ya fueron extraídos para este snapshot.
existing_file_names = {
    file.name
    for file in dbutils.fs.ls(staging_snapshot_path)
}

extraction_rows = []

# Leemos el ZIP desde Raw. Nunca modificamos su contenido.
with open(raw_zip_path, "rb") as raw_zip_file:
    with zipfile.ZipFile(raw_zip_file) as gtfs_zip:

        # GTFS usa archivos de texto. Ignoramos directorios u otros archivos.
        members = [
            member
            for member in gtfs_zip.infolist()
            if not member.is_dir()
            and member.filename.lower().endswith(".txt")
        ]

        # Convertimos rutas internas del ZIP a nombres de archivo seguros.
        target_file_names = [
            PurePosixPath(member.filename).name
            for member in members
        ]

        # Evitamos sobrescribir datos si un ZIP tuviera dos archivos con el mismo nombre.
        if len(target_file_names) != len(set(target_file_names)):
            raise RuntimeError(
                "El ZIP contiene nombres de archivos GTFS duplicados."
            )

        for member, target_file_name in zip(members, target_file_names):
            staging_file_path = (
                f"{staging_snapshot_path}/{target_file_name}"
            )

            file_already_exists = (
                target_file_name in existing_file_names
            )

            # Solo extraemos si Staging aún no tiene esta copia de trabajo.
            if not file_already_exists:
                with gtfs_zip.open(member) as source_file:
                    with open(staging_file_path, "wb") as target_file:
                        shutil.copyfileobj(source_file, target_file)

            extraction_rows.append((
                member.filename,
                target_file_name,
                member.file_size,
                file_already_exists,
                staging_file_path,
            ))

# Usamos un esquema explícito para evitar que Spark infiera tipos ambiguos.
extraction_schema = StructType([
    StructField("source_member_name", StringType(), False),
    StructField("target_file_name", StringType(), False),
    StructField("uncompressed_bytes", LongType(), False),
    StructField("file_already_exists", BooleanType(), False),
    StructField("staging_file_path", StringType(), False),
])

extraction_df = spark.createDataFrame(
    extraction_rows,
    schema=extraction_schema,
)

# Guardamos un manifiesto de la extracción derivada.
extraction_manifest = {
    "extraction_run_id": extraction_run_id,
    "extracted_at_utc": extracted_at_utc.isoformat(),
    "source_raw_zip_path": raw_zip_path,
    "feed_version": feed_version,
    "source_sha256": snapshot_sha256,
    "staging_snapshot_path": staging_snapshot_path,
    "files_extracted": [
        {
            "source_member_name": row[0],
            "target_file_name": row[1],
            "uncompressed_bytes": row[2],
            "file_already_exists": row[3],
        }
        for row in extraction_rows
    ],
}

extraction_manifest_path = (
    f"{staging_snapshot_path}/"
    f"_extraction_run_id={extraction_run_id}.json"
)

dbutils.fs.put(
    extraction_manifest_path,
    json.dumps(extraction_manifest, indent=2),
    overwrite=False,
)

display(extraction_df)

In [0]:
# BLOQUE 3 — Lectura y prevalidación de routes.txt desde Staging

from pyspark.sql import functions as F

# routes.txt fue extraído del ZIP validado en el bloque anterior.
routes_staging_path = f"{staging_snapshot_path}/routes.txt"

# Bronze conserva los campos como texto para no alterar el dato fuente.
# Los casteos de tipos y reglas de negocio ocurrirán más adelante en Silver.
routes_source_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(routes_staging_path)
)

# Agregamos procedencia técnica para mantener trazabilidad del registro.
routes_bronze_preview_df = (
    routes_source_df
    .withColumn("source_feed_version", F.lit(feed_version))
    .withColumn("source_sha256", F.lit(snapshot_sha256))
    .withColumn("source_file_path", F.lit(routes_staging_path))
    .withColumn("bronze_loaded_at_utc", F.current_timestamp())
)

print("Esquema leído desde routes.txt:")
routes_bronze_preview_df.printSchema()

print(f"Cantidad de rutas leídas: {routes_bronze_preview_df.count()}")

display(
    routes_bronze_preview_df
    .orderBy("route_id")
    .limit(20)
)

In [0]:
# BLOQUE 4 — Escritura idempotente de routes.txt en Delta Bronze

bronze_routes_table = (
    f"{catalog}.{schema}.bronze_gtfs_routes"
)

# La primera ejecución crea una tabla Delta vacía con el esquema ya validado.
# Las ejecuciones posteriores reutilizan la misma tabla gobernada por Unity Catalog.
if not spark.catalog.tableExists(bronze_routes_table):
    (
        routes_bronze_preview_df
        .limit(0)
        .write
        .format("delta")
        .saveAsTable(bronze_routes_table)
    )

# Un snapshot GTFS completo se identifica por su hash.
# Delta garantiza escrituras atómicas: evitamos insertar otra vez toda la versión
# si esa misma evidencia ya fue cargada anteriormente.
snapshot_already_loaded = (
    spark.table(bronze_routes_table)
    .filter(F.col("source_sha256") == snapshot_sha256)
    .limit(1)
    .count() > 0
)

if not snapshot_already_loaded:
    (
        routes_bronze_preview_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_routes_table)
    )

bronze_snapshot_df = (
    spark.table(bronze_routes_table)
    .filter(F.col("source_sha256") == snapshot_sha256)
)

print(f"Tabla Bronze: {bronze_routes_table}")
print(f"Snapshot ya cargado antes: {snapshot_already_loaded}")
print(
    "Rutas Bronze para este snapshot: "
    f"{bronze_snapshot_df.count()}"
)

display(
    bronze_snapshot_df
    .orderBy("route_id")
    .limit(20)
)

In [ ]:
# BLOQUE 4A — Auditoría de deriva de esquema entre snapshots de rutas

from pyspark.sql.types import StringType, StructField, StructType

# Las columnas técnicas las agrega el pipeline; la comparación se limita
# a los campos publicados por TransMilenio en routes.txt.
technical_columns = {
    "source_feed_version",
    "source_sha256",
    "source_file_path",
    "bronze_loaded_at_utc",
}
bronze_data_columns = (
    set(spark.table(bronze_routes_table).columns) - technical_columns
)
source_columns = set(routes_source_df.columns)

# Identificamos cambios del proveedor sin añadir valores ni eliminar historia.
missing_in_current_source = sorted(bronze_data_columns - source_columns)
new_in_current_source = sorted(source_columns - bronze_data_columns)

schema_drift_schema = StructType([
    StructField("drift_type", StringType(), False),
    StructField("columns", StringType(), False),
])
schema_drift_rows = [
    (
        "missing_in_current_source",
        ", ".join(missing_in_current_source) or "ninguna",
    ),
    (
        "new_in_current_source",
        ", ".join(new_in_current_source) or "ninguna",
    ),
]

display(
    spark.createDataFrame(
        schema_drift_rows,
        schema=schema_drift_schema,
    )
)

# Esta métrica comprueba cómo la columna histórica se comporta por versión.
# Un valor nulo en 2026 es evidencia de ausencia en el proveedor, no un valor inventado.
display(
    spark.table(bronze_routes_table)
    .groupBy("source_feed_version")
    .agg(
        F.count("*").alias("route_records"),
        F.sum(
            F.when(F.col("route_desc").isNotNull(), 1).otherwise(0)
        ).alias("route_desc_non_null_records"),
    )
    .orderBy("source_feed_version")
)

In [0]:
# Consulta las primeras rutas persistidas en Bronze.
display(
    spark.table("workspace.gtfs_bogota.bronze_gtfs_routes")
    .orderBy("route_id")
    .limit(20)
)

In [0]:
# Muestra formato, ubicación, número de archivos y tamaño de la tabla.
display(
    spark.sql("""
        describe detail workspace.gtfs_bogota.bronze_gtfs_routes
    """)
)

In [0]:
# Muestra las versiones y operaciones realizadas sobre la tabla.
display(
    spark.sql("""
        describe history workspace.gtfs_bogota.bronze_gtfs_routes
    """)
)